# Library Imports & Setup
This section includes the necessary library installations and imports required for molecular docking and dynamics simulations.

### Installing Dependencies
The following commands install required packages using both `pip` and `conda`. They ensure compatibility with cheminformatics and molecular docking tools such as `RDKit`, `OpenMM`, and `AutoDock Vina`.

### Importing Required Libraries
Below, we import key Python libraries:
- `mdtraj`: For trajectory analysis in molecular dynamics.
- `openff.toolkit`: For handling molecular structures and force fields.
- `openmm`: A high-performance toolkit for molecular simulations.
- `rdkit`: For cheminformatics and molecular modeling.
- `plotly`: To generate interactive plots for visualization.



# Installs and dependencies

Create a conda enviroment to handle specific packages like vina, openmm, openbabel, etc. This step takes around 5 seconds and you may see a restarting kernel message that is expected.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:11
🔁 Restarting kernel...


Install `pip` dependencies

⚠ Note: Running the following cell will install the required packages (rdkit and deepchem).
This may restart the Jupyter kernel, which is normal.
If the kernel restarts, simply re-run this cell to continue with workflow.

In [ ]:
!pip install rdkit
!pip install deepchem

  Using cached deepchem-2.8.0-py3-none-any.whl.metadata (2.0 kB)
  Using cached scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (18 kB)
  Using cached sympy-1.13.3-py3-none-any.whl.metadata (12 kB)
  Using cached threadpoolctl-3.5.0-py3-none-any.whl.metadata (13 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached deepchem-2.8.0-py3-none-any.whl (1.0 MB)
Using cached scikit_learn-1.6.1-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (13.5 MB)
Using cached sympy-1.13.3-py3-none-any.whl (6.2 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
Using cached threadpoolctl-3.5.0-py3-none-any.whl (18 kB)


Install `conda` dependencies

In [ ]:
!conda install conda-forge::openbabel
!conda install conda-forge::openmmforcefields
!conda install conda-forge::openmm=7.7
!conda install conda-forge::openff-forcefields=2023.08.0
!conda install conda-forge::openff-toolkit=0.14.3
!conda install conda-forge::parmed=4.1
!conda install conda-forge::pdbfixer=1.8
!conda install conda-forge::mdtraj=1.9.9
!conda install conda-forge::rdkit=2023.03
!conda install conda-forge::plotly=4.9.0
!conda install conda-forge::python-kaleido=0.2.1
!conda install -c conda-forge pdbfixer
!conda install -c conda-forge vina

Streaming output truncated to the last 5000 lines.








                                                                        














                                                                        















                                                                        
















                                                                        

















                                                                        













































Preparing transaction: - done
Verifying transaction: | / - \ | done
Executing transaction: - \ | / - \ | / - \ | / - \ | / done
Channels:
 - conda-forge
Platform: linux-64
Solving environment: | / - \ | / - \ | / done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.3
    latest version: 25.1.1

Please update conda by running

    $ conda update -n base -c conda-forge cond

# Import libraries and classes


In [ ]:
import mdtraj as md
from openff.toolkit.topology import Molecule
from openff.toolkit.typing.engines.smirnoff import ForceField
import openmm.app as app
import openmm as mm
import openmm.unit as unit
from openmmforcefields.generators import SystemGenerator
import sys, time, argparse
from openmm import app, unit, LangevinIntegrator, Vec3, MonteCarloBarostat
from openmm.app import PDBFile, Simulation, Modeller, PDBReporter, StateDataReporter, DCDReporter
from openmm.unit import nanometers
import os
from openmm import Platform
import sys, argparse
import mdtraj as md
import plotly.graph_objects as go
import deepchem as dc

No normalization for SPS. Feature removed!
No normalization for AvgIpc. Feature removed!
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead


wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Skipped loading modules with pytorch-geometric dependency, missing a dependency. No module named 'torch_geometric'
Skipped loading modules with pytorch-geometric dependency, missing a dependency. cannot import name 'DMPNN' from 'deepchem.models.torch_models' (/usr/local/lib/python3.11/site-packages/deepchem/models/torch_models/__init__.py)
Skipped loading modules with pytorch-lightning dependency, missing a dependency. No module named 'lightning'
Skipped loading some Jax models, missing a dependency. No module named 'haiku'


# Molecular Docking

Create a directory to store docking results

In [ ]:
!mkdir results

## Perform molecular docking with AutoDock Vina through Deepchem

To run a molecular docking workflow we need a target (protein file) and a ligand. Also, Autodock Vina employs two parameters, exhaustiveness and num_modes, for computing the protein-ligand interactions. The `exhaustiveness` parameter is related to the computation needed for the calculations, which is directly related to CPUs availavilty. Furthermore, `num_modes` is used to generate a custom amount of number of conformational modes. In the study, exhaustivenees was set to 2, for low-computational cost, and 20 for a heavier computational cost.

In [ ]:
protein_file = "input.pdb"
ligand_file = "ligand.sdf"
exhaustiveness = 2
num_modes=1

In [ ]:
vpg = dc.dock.VinaPoseGenerator()

poses, scores = vpg.generate_poses((protein_file, ligand_file),
                                    exhaustiveness=exhaustiveness,
                                    num_modes=num_modes,
                                    out_dir='results',
                                    generate_scores=True)

In [ ]:
print("The binding affinity score was ",scores[0], " kcal/mol")

The binding affinity score was  -6.072  kcal/mol


Convert docked ligand file from PDBQT to SDF file type needed for molecular dynamics simulations

In [ ]:
!obabel results/ligand_docked.pdbqt -O docked_ligand.sdf
l = "docked_ligand.sdf"

1 molecule converted


# Molecular Dynamics

###Helper Functions for Molecular Dynamics Simulations in OpenMM

In molecular dynamics (MD) simulations, accurate preparation of the system is essential to ensure meaningful results. This section provides utility functions designed to facilitate MD simulations in OpenMM by addressing key preprocessing steps. These functions are specifically tailored to correct ligand representations, optimize simulation settings, and ensure compatibility with force fields such as AMBER and GAFF.



1.   `fix_ligand(input_file, output_file)`
     This function processes small molecule structures stored in Structure Data File (SDF) format to correct atomic charge representations. Ensuring proper atomic charges is crucial for accurate electrostatic interactions during MD simulations. The function modifies only the atomic charge column while preserving bond orders, connectivity, and stereochemistry. This correction prevents incorrect neutralization of charged residues and ensures compatibility with force fields in OpenMM.

2.   `get_platform()`
      OpenMM supports multiple computational platforms, including CUDA (GPU), OpenCL, and CPU, with varying levels of performance. This function automatically selects the optimal computing platform based on availability, prioritizing GPU acceleration when possible. If a GPU is detected, the function sets the floating-point precision to "mixed", optimizing performance without significant loss of accuracy.



These helper functions are designed to streamline system preparation and simulation execution, ensuring efficient and accurate molecular dynamics workflows. They are especially useful when running large-scale biomolecular simulations that require high precision and reproducibility.

In [ ]:
def fix_ligand(input_file, output_file):
    """
    Corrects atomic charges in an SDF ligand file for molecular dynamics (MD) simulations in OpenMM.

    This function processes a ligand structure stored in an SDF file, ensuring that atomic charges
    are properly set for force field compatibility in OpenMM-based MD simulations. It specifically
    modifies the atomic charge column while preserving bond orders and other molecular properties.

    The charge corrections are necessary to:
    - Ensure accurate electrostatic interactions during MD simulations.
    - Prevent incorrect neutralization of charged residues in the ligand.
    - Maintain compatibility with the selected force field (e.g., AMBER, GAFF).

    This function does not alter bond orders, atomic connectivity, or stereochemistry.

    Parameters:
    - input_file (str): Path to the input ligand file.
    - output_file (str): Path to the output ligand file after modifications.

    Returns:
    - Molecule: An OpenFF Molecule object representing the corrected ligand.
    """
    # Open the file in write mode, which creates an empty file
    with open(output_file, 'w') as file:
        pass  # No content is written, so the file will be empty

    with open(input_file, 'r') as infile, open(output_file, 'w', newline='') as outfile:
        in_f = infile.readlines()
        l1 = in_f[:4]
        l3 = in_f[36:]
        outfile.writelines(l1)

        for line in in_f[4:36]:
            if ' 3 ' or ' 2 ' in line:
                line = line.replace(' 3 ', ' 0 ')
                line = line.replace(' 2 ', ' 0 ')
            outfile.write(line)

        outfile.writelines(l3)

    time.sleep(2)
    ligand = Molecule.from_file(output_file)
    return ligand


def get_platform():
    """
    Determines the optimal computing platform for molecular simulations.

    This function retrieves the computing platform for OpenMM simulations, prioritizing
    user-defined platforms if specified via the `PLATFORM` environment variable.
    If no platform is specified, it automatically selects the fastest available option.
    If a GPU-based platform is used, the precision is set to "mixed" for performance optimization.

    Returns:
    - Platform: The selected OpenMM platform for simulation.
    """
    os_platform = os.getenv('PLATFORM')
    if os_platform:
        platform = Platform.getPlatformByName(os_platform)
    else:
        # Automatically select the fastest platform available
        speed = 0
        for i in range(Platform.getNumPlatforms()):
            p = Platform.getPlatform(i)
            if p.getSpeed() > speed:
                platform = p
                speed = p.getSpeed()

    print('Using platform', platform.getName())

    # If it's a GPU platform, set the precision to mixed for better performance
    if platform.getName() in ['CUDA', 'OpenCL']:
        platform.setPropertyDefaultValue('Precision', 'mixed')
        print(f'Set precision for platform {platform.getName()} to mixed')

    return platform

### Run molecular dynamics simulations through openmm

Molecular dynamics (MD) simulations provide valuable insights into the dynamic behavior of biomolecular systems, such as protein-ligand interactions. This script sets up and runs an explicit solvent MD simulation in OpenMM, including system preparation, equilibration, and production runs. The simulation parameters are carefully chosen to ensure stability, accuracy, and computational efficiency.

**Key Parameters and Simulation Time**

The total simulated time and computational parameters are defined as follows:

*   Timestep: 2 fs (0.002 ps)
*   Equilibration Steps: 1,250,000 (~2.5 ns)
*   Production Steps: 5,000,000 (~10 ns)
*   Total Simulated Time: 10 ns
*   Temperature: 300 K (Langevin thermostat)
*   Pressure: 1 atm (Monte Carlo barostat)
*   Force Fields: AMBER14SB, GAFF-2.11
*   Water Model: TIP3P
*   Solvent Box Padding: 2 nm


The simulation uses a Langevin integrator, which ensures accurate temperature control by applying stochastic forces. A Monte Carlo barostat maintains a constant pressure ensemble (NPT) to simulate physiological conditions.

**Workflow of the Script
System Preparation**

The protein and ligand structures are loaded.
The ligand is processed using the `fix_ligand()` function to ensure correct atomic charges.
Hydrogen atoms are added to the system.
The protein-ligand complex is solvated in a periodic water box with 0.15 M NaCl.

**GPU availabilty**

Since running 10 ns of simulation on a CPU would be computationally expensive, this version of the notebook was executed for 10 ps (equivalent to 5 000 steps) for computational time purposes. The described simulation parameters and methodology, however, are intended for a 10 ns production run.

This adjustment allows for demonstration and validation of the workflow without requiring excessive computation time. The results remain valid for larger-scale simulations, which can be executed on high-performance computing (HPC) clusters with GPU acceleration.

In [ ]:
t0 = time.time()

protein_pdb = app.PDBFile(protein_file)

try:
    ligand = Molecule.from_file(l)
except:
    ligand = fix_ligand(l, l[:-4]+'_fixed.sdf')


forcefield_kwargs = {'constraints': app.HBonds, 'rigidWater': True, 'removeCMMotion': False, 'hydrogenMass': 4*unit.amu }
system_generator = SystemGenerator(
      forcefields=['amber14-all.xml', 'amber14/tip3pfb.xml'],
      small_molecule_forcefield='gaff-2.11',
      molecules=[ligand],
      forcefield_kwargs=forcefield_kwargs)

modeller = Modeller(protein_pdb.topology, protein_pdb.positions)
modeller.addHydrogens()
lig_top = ligand.to_topology()

modeller.add(lig_top.to_openmm(), lig_top.get_positions().to_openmm())
print('System has %d atoms' % modeller.topology.getNumAtoms())

waterBox = tuple((max((pos[i] for pos in modeller.positions))-min((pos[i] for pos in modeller.positions))).value_in_unit(nanometers) for i in range(3))

modeller.addSolvent(system_generator.forcefield, model="tip3p", boxSize=(waterBox[0] + 2, waterBox[1] + 2, waterBox[2] + 2),
                          positiveIon="Na+", negativeIon="Cl-",
                          ionicStrength=0.15 * unit.molar, neutralize=True)
print('System has %d atoms' % modeller.topology.getNumAtoms())


system = system_generator.create_system(modeller.topology, molecules=ligand)


friction_coeff = 1 / unit.picosecond
step_size = 0.002 * unit.picoseconds

num_steps = 5000
# num_steps = 5000000 # for 10 ns total simulation time
duration = (step_size * num_steps).value_in_unit(unit.nanoseconds)
print('Simulating for {} ns'.format(duration))

temperature = 300 * unit.kelvin
integrator = LangevinIntegrator(temperature, friction_coeff, step_size)
system.addForce(MonteCarloBarostat(1 * unit.atmospheres, temperature, 25))

print('Default Periodic box: {}'.format(system.getDefaultPeriodicBoxVectors()))
platform = get_platform()
simulation = Simulation(modeller.topology, system, integrator, platform=platform)
context = simulation.context

context.setPositions(modeller.positions
                       )


System has 6919 atoms
System has 47266 atoms
Simulating for 0.01 ns
Default Periodic box: [Quantity(value=Vec3(x=7.9865, y=0.0, z=0.0), unit=nanometer), Quantity(value=Vec3(x=0.0, y=7.8266, z=0.0), unit=nanometer), Quantity(value=Vec3(x=0.0, y=0.0, z=8.1236), unit=nanometer)]
Using platform CPU


**System Initialization and Minimization**

The force field is applied using AMBER14SB (protein) and GAFF-2.11 (ligand).
The system undergoes energy minimization to remove steric clashes.
The potential energy before and after minimization is recorded.
Equilibration (1.25 Million Steps = 2.5 ns)

The system is equilibrated under NPT conditions to reach a stable thermodynamic state.
Temperature and potential energy are monitored in real-time.
Production Simulation (5 Million Steps = 10 ns)

The trajectory is recorded every 10 ps for post-processing.
StateDataReporter logs temperature, energy, and step count.



In [ ]:
state_initial = context.getState(getEnergy=True)
initial_energy = state_initial.getPotentialEnergy()
print(f"Initial Energy (Before Minimization): {initial_energy}")

simulation.minimizeEnergy()

state_minimized = context.getState(getEnergy=True)
minimized_energy = state_minimized.getPotentialEnergy()
print(f"Minimized Energy (After Minimization): {minimized_energy}")


output_base = 'output'
output_complex = output_base + '_complex.pdb'
output_traj_dcd = output_base + '_traj.dcd'
output_min = output_base + '_minimised.pdb'

# Save the initial structure
with open(output_complex, 'w') as outfile:
      PDBFile.writeFile(modeller.topology, modeller.positions, outfile)

with open(output_min, 'w') as outfile:
      PDBFile.writeFile(modeller.topology, context.getState(getPositions=True, enforcePeriodicBox=True).getPositions(), file=outfile, keepIds=True)

simulation.context.setVelocitiesToTemperature(temperature)

print("Equilibration...")
equilibration_steps = 1250
# equilibration_steps = 1250000 # for 10 ns total simulation time
simulation.step(equilibration_steps)

# Set up reporters to log data and write trajectory
reporting_interval = 50
# reporting_interval = 5000  # For 10 ns total simulation time. Save trajectory frames every 10 ps (5,000 steps)
simulation.reporters.append(DCDReporter(output_traj_dcd, reporting_interval, enforcePeriodicBox=True))
simulation.reporters.append(StateDataReporter(sys.stdout, reporting_interval * 2, step=True, potentialEnergy=True, temperature=True))

t1 = time.time()

# Production run: 1 ns (500,000 steps)
simulation.step(num_steps)

t2 = time.time()
print('Simulation complete in {} mins at {}. Total wall clock time was {} mins'.format(
      round((t2 - t1) / 60, 3), temperature, round((t2 - t0) / 60, 3)))



Initial Energy (Before Minimization): -258568.33494067146 kJ/mol
Minimized Energy (After Minimization): -870746.8772330838 kJ/mol
Equilibration...
#"Step","Potential Energy (kJ/mole)","Temperature (K)"
1300,-752029.4369868635,289.58359674622784
1400,-750220.3464478733,291.1054020488083
1500,-749214.435640902,293.3766090721749
1600,-748509.1501473262,294.14521079029845
1700,-747286.1019101557,293.18499708550775
1800,-748027.7273600299,296.2906947025674
1900,-746613.7318346846,295.79823008346887
2000,-745893.0153180499,294.6992270375591
2100,-746502.4513846091,296.2637156661903
2200,-745928.7616722243,298.3457653095659
2300,-745727.5232272899,298.98598447502354
2400,-745103.3281585738,295.71305668677905
2500,-746952.4404005281,298.49133948295344
2600,-746419.8007388385,297.1002230314963
2700,-745938.0192724809,297.07709827304717
2800,-744975.0186139713,296.5040212012897
2900,-745780.5228377559,300.1216465484361
3000,-745809.6315920746,300.6548181225308
3100,-744178.9390489289,297.5047291

**Post-Processing and Analysis**

The trajectory is aligned to remove artifacts due to periodic boundary conditions.
Root Mean Square Deviation (RMSD), Root Mean Square Fluctiations (RMSF), and Radius of Gyration (Rg) calculations are performed for the ligand and protein backbone.
The trajectory and final system configuration are saved.

**Output Files**
* Trajectory (.dcd): Contains the full simulation trajectory for visualization.
* Minimized Structure (.pdb): Energy-minimized protein-ligand complex.
* Aligned Trajectory (.dcd, .pdb): Recentered trajectory after imaging.
* RMSD, RMSF and Rg Plots (.svg): RMSD, RMSF and Rg  analysis of ligand and protein backbone.

This workflow provides a robust and reproducible approach to protein-ligand molecular dynamics simulations in OpenMM. The chosen parameters ensure biophysically meaningful results while maintaining computational efficiency.

In [ ]:
traj_in = output_traj_dcd
topol_in = output_min
out_base = 'output-aligned'

print('Reading trajectory', traj_in)
t = md.load(traj_in, top=topol_in)
t.image_molecules(inplace=True)


print('Realigning')
prot = t.top.select('protein')
t.superpose(t[0], atom_indices=prot)

print('Writing re-imaged PDB', out_base + '.pdb')
t[0].save(out_base + '.pdb')

print('Writing re-imaged trajectory', out_base + '.dcd')
t.save(out_base + '.dcd')

topology = t.topology
print('Number of frames:', t.n_frames)



atoms = t.topology.select("chainid 1")
print(len(atoms), 'ligand atoms')
rmsds_lig = md.rmsd(t, t, frame=0, atom_indices=atoms, parallel=True, precentered=False)


atoms = t.topology.select("chainid 0 and backbone")
print(len(atoms), 'backbone atoms')
rmsds_bck = md.rmsd(t, t, frame=0, atom_indices=atoms, parallel=True, precentered=False)

fig = go.Figure()
fig.add_trace(go.Scatter(x=t.time, y=rmsds_lig, mode='lines', name='Ligand'))
fig.add_trace(go.Scatter(x=t.time, y=rmsds_bck, mode='lines', name='Backbone'))

fig.update_layout(title='Trajectory for ' + traj_in, xaxis_title='Frame', yaxis_title='RMSD')

file = out_base + '.svg'
print('Writing RMSD output to', file)
fig.write_image(file)

Reading trajectory output_traj.dcd
Realigning
Writing re-imaged PDB output-aligned.pdb
Writing re-imaged trajectory output-aligned.dcd
Number of frames: 100
36 ligand atoms
1784 backbone atoms
Writing RMSD output to output-aligned.svg
